# Data Aggregation

> **Definition — aggregation**: a data transformation that produces a *scalar* from an array. `mean`, `count`, `min`, and `sum` are all aggregations — each collapses a group's worth of rows down to a single number per column.

When you call `.mean()` (or any of the methods below) on a GroupBy object, pandas isn't calling generic Python code per group — it dispatches to a **cython-optimized implementation** built specifically for that operation. That's why these are fast even on large datasets:

| Method | What it computes |
|---|---|
| `count` | Number of non-NA values in the group |
| `sum` | Sum of non-NA values |
| `mean` | Mean of non-NA values |
| `median` | Arithmetic median of non-NA values |
| `std`, `var` | Unbiased (n-1 denominator) standard deviation and variance |
| `min`, `max` | Minimum and maximum of non-NA values |
| `prod` | Product of non-NA values |
| `first`, `last` | First and last non-NA values |

## Beyond the built-ins

You aren't limited to this table. You can also call:
- **Any method already defined on the object being grouped** — e.g. a Series' `nsmallest` method — even though GroupBy doesn't special-case it.
- **Your own custom function**, via `.agg()` (short for `.aggregate()`).

Neither of these gets the cython fast path: internally, GroupBy slices the data into per-group pieces, calls your function (or method) on each piece in a Python loop, and glues the results back together. That makes them much more flexible, but slower than the table above for large numbers of groups — reach for a built-in first when one exists.

In [1]:
import numpy as np 
import pandas as pd 


In [2]:
df = pd.DataFrame({"key1" : ["a", "a", None, "b", "b", "a", None],
                "key2" : pd.Series([1, 2, 1, 2, 1, None, 1], dtype="Int64"),
                "data1": np.random.standard_normal(7),
                "data2": np.random.standard_normal(7)})


df

,key1,key2,data1,data2
0,a,1,-0.066109,1.617839
1,a,2,0.507303,-0.553830
2,NaN,1,0.274276,-0.827126
3,b,2,0.523531,0.061837
4,b,1,-1.001738,-0.234402
5,a,<NA>,-0.327609,-1.685495
6,NaN,1,0.005810,0.738749


## Using a Non-Aggregation Method via GroupBy

`nsmallest` selects the *n* smallest values from a Series — it's not an aggregation in the strict sense (it returns multiple rows per group, not one scalar), and GroupBy doesn't implement it directly. But because it's a normal Series method, GroupBy can still run it per group by looping under the hood:

In [3]:
grouped = df.groupby("key1")

print(grouped["data1"].nsmallest(2))



key1   
a     5   -0.327609
      0   -0.066109
b     4   -1.001738
      3    0.523531
Name: data1, dtype: float64


## Custom Aggregations with `agg`

To use your own aggregation function, pass any function that reduces an array to a scalar to the `aggregate` method, or its short alias `agg`:

In [4]:
def peak_to_peak(arr):
    return arr.max() - arr.min()


grouped.agg(peak_to_peak)

,key2,data1,data2
key1,,,
a,1,0.834912,3.303334
b,1,1.525269,0.296240


`peak_to_peak` isn't one of the optimized methods from the table above, so pandas falls back to the slower per-group Python loop described earlier — worth knowing if you're deciding between a custom function and, say, computing `max() - min()` some other way on a very large DataFrame.

You may notice that some methods, like `describe`, also work, even though they aren't aggregations, strictly speaking — `describe` returns *several* values (count, mean, std, quartiles, ...) per group rather than one. GroupBy handles this by computing `describe()` on each group's data and concatenating the results, giving you a wide summary table with hierarchical columns:

In [5]:
print(grouped.describe())

      key2                                           data1            ...  \
     count mean       std  min   25%  50%   75%  max count      mean  ...   
key1                                                                  ...   
a      2.0  1.5  0.707107  1.0  1.25  1.5  1.75  2.0   3.0  0.037862  ...   
b      2.0  1.5  0.707107  1.0  1.25  1.5  1.75  2.0   2.0 -0.239104  ...   

                         data2                                          \
           75%       max count      mean       std       min       25%   
key1                                                                     
a     0.220597  0.507303   3.0 -0.207162  1.678731 -1.685495 -1.119662   
b     0.142213  0.523531   2.0 -0.086283  0.209473 -0.234402 -0.160342   

                                    
           50%       75%       max  
key1                                
a    -0.553830  0.532004  1.617839  
b    -0.086283 -0.012223  0.061837  

[2 rows x 24 columns]


## Column-Wise and Multiple Function Application

Let's switch to the classic **tips** dataset — one row per restaurant bill, with the total bill, tip amount, party size, and some categorical info (day, time, whether the payer was a smoker). This time we'll add a `tip_pct` column (tip as a fraction of the bill) so we have a more interesting numeric column to aggregate:

In [6]:
tips = pd.read_csv("/Users/bmart231/DS/examples/tips.csv")

tips.head()

,total_bill,tip,smoker,day,time,size
0,16.99,1.01,No,Sun,Dinner,2
1,10.34,1.66,No,Sun,Dinner,3
2,21.01,3.50,No,Sun,Dinner,3
3,23.68,3.31,No,Sun,Dinner,2
4,24.59,3.61,No,Sun,Dinner,4


Now add a `tip_pct` column with the tip percentage of the total bill:

In [7]:
tips["tip_pct"] = tips["tip"] / tips["total_bill"]

tips.head()

,total_bill,tip,smoker,day,time,size,tip_pct
0,16.99,1.01,No,Sun,Dinner,2,0.059447
1,10.34,1.66,No,Sun,Dinner,3,0.160542
2,21.01,3.50,No,Sun,Dinner,3,0.166587
3,23.68,3.31,No,Sun,Dinner,2,0.139780
4,24.59,3.61,No,Sun,Dinner,4,0.146808


In [8]:
grouped = tips.groupby(["day", "smoker"])

grouped_pct = grouped["tip_pct"]

grouped_pct.agg("mean")

day   smoker
Fri   No        0.151650
      Yes       0.174783
Sat   No        0.158048
      Yes       0.147906
Sun   No        0.160113
      Yes       0.187250
Thur  No        0.160298
      Yes       0.163863
Name: tip_pct, dtype: float64

Passing the string `"mean"` to `.agg()` is equivalent to calling `.mean()` directly — the string is just looked up as a method name. This matters because it means you can mix string names and real functions freely, as shown next.

If you pass a list of functions or function names instead, you get back a DataFrame with one column per function, and the column names are taken from each function's name (`peak_to_peak.__name__`, in this case) — which is why using a custom function with a descriptive name pays off:

In [9]:
grouped_pct.agg(["mean", "std", peak_to_peak])

mean       std  peak_to_peak
day  smoker                                  
Fri  No      0.151650  0.028123      0.067349
     Yes     0.174783  0.051293      0.159925
Sat  No      0.158048  0.039767      0.235193
     Yes     0.147906  0.061375      0.290095
Sun  No      0.160113  0.042347      0.193226
     Yes     0.187250  0.154134      0.644685
Thur No      0.160298  0.038774      0.193350
     Yes     0.163863  0.039389      0.151240

If you'd rather control the column names yourself, pass a list of `(name, function)` tuples instead — the first element of each tuple becomes the column name:

In [10]:
grouped_pct.agg([("average", "mean"), ("stdev", np.std)])

average     stdev
day  smoker                    
Fri  No      0.151650  0.024355
     Yes     0.174783  0.049553
Sat  No      0.158048  0.039323
     Yes     0.147906  0.060640
Sun  No      0.160113  0.041974
     Yes     0.187250  0.150023
Thur No      0.160298  0.038341
     Yes     0.163863  0.038213

With a DataFrame you have more options, as you can specify a list of functions to apply to all of the columns or different functions per column. To start, suppose we wanted to compute the same three statistics for the `tip_pct` and `total_bill` columns. 

In [11]:
functions = ["count", "mean", "max"]

result = grouped[["tip_pct", "total_bill"]].agg(functions)

result

tip_pct                     total_bill                  
              count      mean       max      count       mean    max
day  smoker                                                         
Fri  No           4  0.151650  0.187735          4  18.420000  22.75
     Yes         15  0.174783  0.263480         15  16.813333  40.17
Sat  No          45  0.158048  0.291990         45  19.661778  48.33
     Yes         42  0.147906  0.325733         42  21.276667  50.81
Sun  No          57  0.160113  0.252672         57  20.506667  48.17
     Yes         19  0.187250  0.710345         19  24.120000  45.35
Thur No          45  0.160298  0.266312         45  17.113111  41.19
     Yes         17  0.163863  0.241255         17  19.190588  43.11

As you can see, the resulting DataFrame has hierarchial columns, the same as you would get aggregating each column separately and using `concat` to glue the results together using the columns names as the `keys` argument:

In [12]:
result["tip_pct"]

count      mean       max
day  smoker                           
Fri  No          4  0.151650  0.187735
     Yes        15  0.174783  0.263480
Sat  No         45  0.158048  0.291990
     Yes        42  0.147906  0.325733
Sun  No         57  0.160113  0.252672
     Yes        19  0.187250  0.710345
Thur No         45  0.160298  0.266312
     Yes        17  0.163863  0.241255

As before, a list of tuples with custom names can be passed:

In [13]:
ftuples = [("Average", "mean"), ("Variance", np.var)]

grouped[["tip_pct", "total_bill"]].agg(ftuples)

tip_pct           total_bill            
              Average  Variance    Average    Variance
day  smoker                                           
Fri  No      0.151650  0.000593  18.420000   19.197250
     Yes     0.174783  0.002456  16.813333   77.058276
Sat  No      0.158048  0.001546  19.661778   78.133210
     Yes     0.147906  0.003677  21.276667   98.973546
Sun  No      0.160113  0.001762  20.506667   64.940331
     Yes     0.187250  0.022507  24.120000  103.306779
Thur No      0.160298  0.001470  17.113111   58.300079
     Yes     0.163863  0.001460  19.190588   65.702135

## Applying Different Functions to Different Columns

Now, suppose you wanted to apply potentially different functions to one or more of the columns. To do this, pass a dict to `agg` that maps column names to any of the function specifications listed so far:

In [14]:
grouped.agg({"tip_pct": "min", "size": "sum"})

tip_pct  size
day  smoker                
Fri  No      0.120385     9
     Yes     0.103555    31
Sat  No      0.056797   115
     Yes     0.035638   104
Sun  No      0.059447   167
     Yes     0.065660    49
Thur No      0.072961   112
     Yes     0.090014    40

**Why the dict form is genuinely useful (not just shorter):** `size` here is a *column* name in `tips`, but `size` is also a GroupBy *method* (`grouped.size()`, which counts rows per group — see the previous notebook). Writing `grouped.agg({"size": "sum"})` disambiguates this: it's explicit that you mean "sum the `size` column," not "call the `.size()` method."

Each column can also take a *list* of functions, not just one — mix and match freely:

In [15]:
grouped.agg({"tip_pct": ["min", "max", "mean", "std"], "size": "sum"})

tip_pct                               size
                  min       max      mean       std  sum
day  smoker                                             
Fri  No      0.120385  0.187735  0.151650  0.028123    9
     Yes     0.103555  0.263480  0.174783  0.051293   31
Sat  No      0.056797  0.291990  0.158048  0.039767  115
     Yes     0.035638  0.325733  0.147906  0.061375  104
Sun  No      0.059447  0.252672  0.160113  0.042347  167
     Yes     0.065660  0.710345  0.187250  0.154134   49
Thur No      0.072961  0.266312  0.160298  0.038774  112
     Yes     0.090014  0.241255  0.163863  0.039389   40

## Returning Aggregated Data Without a Row Index

In all of the examples so far, the aggregated result comes back with the group keys (`day`, `smoker`) as a hierarchical **row index** rather than as regular columns. That's not always what you want — e.g. if you plan to feed the result into another `merge` or just want a "flat" table. Pass `as_index=False` to `groupby` to keep the group keys as ordinary columns instead:

In [16]:
tips.groupby(["day", "smoker"], as_index=False).mean(numeric_only=True)

,day,smoker,total_bill,tip,size,tip_pct
0,Fri,No,18.420000,2.812500,2.250000,0.151650
1,Fri,Yes,16.813333,2.714000,2.066667,0.174783
2,Sat,No,19.661778,3.102889,2.555556,0.158048
3,Sat,Yes,21.276667,2.875476,2.476190,0.147906
4,Sun,No,20.506667,3.167895,2.929825,0.160113
5,Sun,Yes,24.120000,3.516842,2.578947,0.187250
6,Thur,No,17.113111,2.673778,2.488889,0.160298
7,Thur,Yes,19.190588,3.030000,2.352941,0.163863


This is equivalent to calling `.reset_index()` on the indexed result — `as_index=False` just saves you the extra step, and avoids doing the (usually cheap, but not free) work of building the hierarchical index in the first place.

---

## Summary / Cheat Sheet

**Ways to run an aggregation:**

| What you want | How |
|---|---|
| One built-in stat, all columns | `grouped.mean()`, `grouped.sum()`, ... |
| One built-in stat, one column | `grouped["col"].mean()` |
| A method not built into GroupBy | `grouped["col"].nsmallest(2)` (works via a per-group Python loop) |
| A custom function | `grouped.agg(my_func)` |
| Several functions at once | `grouped["col"].agg(["mean", "std", my_func])` |
| Several functions with custom names | `grouped["col"].agg([("avg", "mean"), ("var", np.var)])` |
| Different function(s) per column | `grouped.agg({"col1": "sum", "col2": ["min", "max"]})` |
| Flat result (no index from group keys) | `df.groupby(keys, as_index=False).mean()` |

**Nuances worth remembering:**
- Methods in the built-in table (`mean`, `sum`, `count`, ...) are cython-optimized and fast. Anything else — a custom function, or a non-aggregating method like `nsmallest`/`describe` — runs as a Python loop over each group, which is much slower on many groups.
- A string like `"mean"` passed to `.agg()` is just looked up as a method name — `.agg("mean")` and `.mean()` do the same thing.
- Passing a list of functions/strings to `.agg()` names each output column after the function (`__name__` for custom functions) unless you supply `(name, function)` tuples instead.
- The dict form of `.agg()` isn't just shorthand — it's the only way to unambiguously target a *column* whose name collides with a GroupBy *method* name (e.g. a `size` column vs. the `.size()` method).